# Program 07: the retrain baseline (the gold standard of unlearning)

**What:** train a model from scratch on all CIFAR-10 classes **except one** (class 3, "cat"). Save it as `results/retrain_model.pt`.

**How:** program 01's training recipe (ResNet-18, SGD, cosine learning rate, 30 epochs), with every cat image removed from the training data.

**Why:** exact unlearning means "behave as if never trained on that data". The retrained model **is** that ideal, and every unlearning method (programs 08, 09, and our own later) is judged by how close it gets to it.

**Understanding note:** retraining is perfect but too expensive to repeat on every deletion request. All approximate unlearning methods trade some of that perfection for speed.

**Done when:** cat accuracy is 0% (the class was never learned) and the other classes are near program 01.

## Phase C in short (for anyone reading this code for the first time)

Phases A and B were about **continual learning**: learning new tasks without forgetting old ones. Phase C switches to **unlearning**: forgetting something *on purpose*. We start simple, with one fully trained model on CIFAR-10 and one class to forget, before mixing unlearning with continual learning in Phase D.

Words used from here on:

| Term | Meaning |
|---|---|
| **forget set** | the data we are asked to remove: here, every cat image |
| **retain set** | everything else: the other 9 classes |
| **original model** | trained on all data, cats included (program 01) |
| **retrain model** | trained from scratch on the retain set only (this program) |
| **FA**, forget accuracy | accuracy on the forget set's test images. Retrain model: 0% |
| **RA**, retain accuracy | accuracy on the retain set's test images. Should stay high |
| **exact unlearning** | the result is identical to retraining without the data |
| **approximate unlearning** | cheaper methods that try to *look like* the retrain model (programs 08, 09) |

So this program builds the **target** that the next three programs aim for.

## 1. Setup
Same training settings as program 01, so the only difference between the two models is the missing class.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import common/

# Reload the shared code, so edits to common/ are picked up even without restarting the kernel.
import importlib
import common.utils, common.data, common.models, common.train
for module in (common.utils, common.data, common.models, common.train):
    importlib.reload(module)

import matplotlib.pyplot as plt
import numpy as np
import torch

from common.data import get_cifar10_unlearning_loaders
from common.models import resnet18_cifar
from common.train import evaluate, per_class_accuracy, predict_all, train_one_epoch
from common.utils import RESULTS_DIR, get_device, log_run, set_seed

SETTINGS = {
    "forget_class": 3,       # CIFAR-10 class 3 = cat
    "epochs": 30, "lr": 0.1, "batch_size": 128, "weight_decay": 5e-4,
    "seed": 0, "workers": 4,
}
set_seed(SETTINGS["seed"])
device = get_device()
print("device:", device)

device: cuda


## 2. Data: split CIFAR-10 into forget and retain sets

`get_cifar10_unlearning_loaders` (in `common/data.py`) does the split once; programs 08 to 10 reuse it. The first run downloads CIFAR-10 (about 170 MB) into `data/`.

We expect 45,000 retain + 5,000 forget training images, and 9,000 + 1,000 test images.

In [2]:
loaders = get_cifar10_unlearning_loaders(SETTINGS["forget_class"], SETTINGS["batch_size"], SETTINGS["workers"])
classes = loaders["classes"]
FORGET = SETTINGS["forget_class"]

for name in ["train_retain", "train_forget", "test_retain", "test_forget"]:
    print(f"{name:13} {len(loaders[name].dataset):6} images")
print(f"\nforget class: {FORGET} ({classes[FORGET]})")

# Check: no cat in the retain set (read the stored labels, no image loading).
retain_sub = loaders["train_retain"].dataset
assert all(retain_sub.dataset.targets[i] != FORGET for i in retain_sub.indices), "a cat leaked into the retain set"
print("retain set has no cat images: OK")

train_retain   45000 images
train_forget    5000 images
test_retain     9000 images
test_forget     1000 images

forget class: 3 (cat)
retain set has no cat images: OK


## 3. The model: still 10 outputs

The model keeps **all 10 outputs**, including one for cat, even though it never sees a cat. Two reasons:
- It must be directly comparable with the original model: same architecture, same output meaning, so later programs can compare them output by output.
- An unlearned model in practice still has the cat output; unlearning doesn't change the architecture.

The cat output is never the correct answer during training, so cross-entropy only ever pushes it **down**. That is why we expect cat accuracy to be exactly 0%.

In [3]:
model = resnet18_cifar(num_classes=10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=SETTINGS["lr"], momentum=0.9,
                            weight_decay=SETTINGS["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS["epochs"])
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 4. Train on the retain set only

Program 01's loop, using only `train_retain`. After each epoch we measure:
- **RA**: accuracy on the 9,000 retain test images (should climb to program 01's level).
- **FA**: accuracy on the 1,000 cat test images (should stay at 0%).

About 20 to 30 seconds per epoch on the laptop GPU, so roughly 12 to 15 minutes in total.

In [4]:
history = {"train_loss": [], "train_acc": [], "ra": [], "fa": []}
start = time.time()
for epoch in range(1, SETTINGS["epochs"] + 1):
    loss, train_acc = train_one_epoch(model, loaders["train_retain"], optimizer, scaler, device)
    ra = evaluate(model, loaders["test_retain"], device)
    fa = evaluate(model, loaders["test_forget"], device)
    scheduler.step()
    for k, v in zip(history, [loss, train_acc, ra, fa]):
        history[k].append(v)
    print(f"epoch {epoch:2d}/{SETTINGS['epochs']}  loss {loss:.3f}  train {train_acc*100:5.1f}%  "
          f"RA {ra*100:5.1f}%  FA {fa*100:4.1f}%  ({time.time()-start:.0f}s)")
train_minutes = (time.time() - start) / 60   # training time only, used in section 9

epoch  1/30  loss 1.835  train  33.8%  RA  42.6%  FA  0.0%  (57s)
epoch  2/30  loss 1.301  train  51.9%  RA  51.4%  FA  0.0%  (83s)
epoch  3/30  loss 1.002  train  64.2%  RA  66.2%  FA  0.0%  (109s)
epoch  4/30  loss 0.771  train  72.8%  RA  72.6%  FA  0.0%  (135s)
epoch  5/30  loss 0.609  train  78.9%  RA  80.4%  FA  0.0%  (161s)
epoch  6/30  loss 0.521  train  82.0%  RA  81.2%  FA  0.0%  (187s)
epoch  7/30  loss 0.464  train  83.9%  RA  78.1%  FA  0.0%  (213s)
epoch  8/30  loss 0.420  train  85.5%  RA  83.7%  FA  0.0%  (238s)
epoch  9/30  loss 0.387  train  86.7%  RA  77.1%  FA  0.0%  (264s)
epoch 10/30  loss 0.361  train  87.6%  RA  86.0%  FA  0.0%  (290s)
epoch 11/30  loss 0.343  train  88.3%  RA  85.6%  FA  0.0%  (316s)
epoch 12/30  loss 0.321  train  88.9%  RA  83.0%  FA  0.0%  (342s)
epoch 13/30  loss 0.302  train  89.6%  RA  85.2%  FA  0.0%  (368s)
epoch 14/30  loss 0.280  train  90.5%  RA  85.4%  FA  0.0%  (394s)
epoch 15/30  loss 0.254  train  91.3%  RA  86.7%  FA  0.0%  (419

## 5. Save the model, and check that reloading gives the same numbers

Programs 08 to 10 load this file as the target to compare against.

In [5]:
model_path = RESULTS_DIR / "retrain_model.pt"
RESULTS_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), model_path)

reloaded = resnet18_cifar(num_classes=10).to(device)
reloaded.load_state_dict(torch.load(model_path, map_location=device))
ra_re = evaluate(reloaded, loaders["test_retain"], device)
fa_re = evaluate(reloaded, loaders["test_forget"], device)
print(f"saved to {model_path}")
print(f"reloaded: RA {ra_re*100:.2f}%  FA {fa_re*100:.2f}%  "
      f"({'same' if abs(ra_re - history['ra'][-1]) < 1e-9 and abs(fa_re - history['fa'][-1]) < 1e-9 else 'DIFFERENT, check this'})")

saved to /run/media/adarsh/AAE039EFE039C1FD/std/research/results/retrain_model.pt
reloaded: RA 94.94%  FA 0.00%  (same)


## 6. Per-class accuracy: retrain model vs original model

The "Done when" check: cat at 0%, the other 9 classes close to program 01.

This needs program 01's saved model (`results/01_resnet18_cifar10.pt`). If it isn't there, this cell says so and only shows the retrain model. To create it, run from the project root: `python programs/01_train_basic.py` (about 15 minutes). Programs 08 to 10 need that file too, because it is the model they unlearn from.

In [ ]:
acc_retrain = per_class_accuracy(model, loaders["test_all"], device)
original_path = RESULTS_DIR / "01_resnet18_cifar10.pt"
acc_original = None
if original_path.exists():
    original = resnet18_cifar(num_classes=10).to(device)
    original.load_state_dict(torch.load(original_path, map_location=device))
    acc_original = per_class_accuracy(original, loaders["test_all"], device)
else:
    print(f"{original_path.name} not found: showing the retrain model only. Run programs/01_train_basic.py to create it.\n")

print(f"{'class':>12} {'retrain':>9}" + (f" {'original':>9}" if acc_original is not None else ""))
for c in range(10):
    mark = "  <- forget class" if c == FORGET else ""
    row = f"{classes[c]:>12} {acc_retrain[c]*100:8.1f}%"
    if acc_original is not None:
        row += f" {acc_original[c]*100:8.1f}%"
    print(row + mark)

retain_mask = np.arange(10) != FORGET
print(f"\nretrain model:  RA {acc_retrain[retain_mask].mean()*100:.1f}%  FA {acc_retrain[FORGET]*100:.1f}%")
if acc_original is not None:
    print(f"original model: RA {acc_original[retain_mask].mean()*100:.1f}%  FA {acc_original[FORGET]*100:.1f}%")

x = np.arange(10)
width = 0.4
plt.figure(figsize=(10, 4))
plt.bar(x - width / 2 if acc_original is not None else x, acc_retrain * 100, width, label="retrain (07)")
if acc_original is not None:
    plt.bar(x + width / 2, acc_original * 100, width, label="original (01)")
plt.xticks(x, classes, rotation=30)
plt.ylabel("test accuracy (%)")
plt.title(f"Per-class accuracy (forget class: {classes[FORGET]})")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "07_per_class.png", dpi=120)
plt.show()

## 7. What does the ideal model do with a cat image?

FA = 0% only says the retrain model never answers "cat". It doesn't say what it answers instead. That matters a lot for the next programs: a good unlearning method should make the original model behave **like this**, not just "never say cat".

We look at two things on the 1,000 cat test images:
1. **Which class it predicts.** We expect mostly visually similar animals (dog, then maybe deer, frog, bird), not a uniform spread. A model that never saw cats still sees "a furry four-legged animal".
2. **How confident it is.** On classes it was trained on, a well-trained model is usually very confident. On cats, which it never saw, we expect lower confidence. This difference is exactly what program 10's membership inference attack looks for: if an unlearned model is still *very* confident on cats, it probably still remembers them.

In [ ]:
probs_forget, _ = predict_all(model, loaders["test_forget"], device)
probs_retain, labels_retain = predict_all(model, loaders["test_retain"], device)

pred = probs_forget.argmax(1)
share = torch.bincount(pred, minlength=10).float() / len(pred)
print(f"retrain model on {classes[FORGET]} test images, predicted as:")
for c in share.argsort(descending=True).tolist():
    if share[c] > 0:
        print(f"  {classes[c]:>12}: {share[c]*100:5.1f}%")

conf_forget = probs_forget.max(1).values
conf_retain = probs_retain.max(1).values
print(f"\naverage top-class confidence: retain test images {conf_retain.mean()*100:.1f}%, "
      f"{classes[FORGET]} test images {conf_forget.mean()*100:.1f}%")
print(f"average probability given to '{classes[FORGET]}' on {classes[FORGET]} images: "
      f"{probs_forget[:, FORGET].mean()*100:.3f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(classes, share.numpy() * 100)
axes[0].set(title=f"Retrain model: predictions on {classes[FORGET]} images", ylabel="% of images")
axes[0].tick_params(axis="x", rotation=30)
bins = np.linspace(0, 1, 21)
axes[1].hist(conf_retain.numpy(), bins=bins, alpha=0.6, density=True, label="retain test images")
axes[1].hist(conf_forget.numpy(), bins=bins, alpha=0.6, density=True, label=f"{classes[FORGET]} test images")
axes[1].set(title="Top-class confidence", xlabel="max softmax probability", ylabel="density")
axes[1].legend()
fig.tight_layout()
plt.show()

# Saved for programs 08-10: the target prediction pattern on cat images.
np.savez(RESULTS_DIR / "07_retrain_reference.npz", per_class_acc=acc_retrain,
         forget_pred_share=share.numpy(), forget_conf=conf_forget.numpy(), retain_conf=conf_retain.numpy())

## 8. Training curves

In [ ]:
ep = range(1, SETTINGS["epochs"] + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ep, history["train_loss"])
ax1.set(title="Training loss (retain set)", xlabel="epoch", ylabel="cross-entropy")
ax2.plot(ep, [a * 100 for a in history["ra"]], label="RA (retain test)")
ax2.plot(ep, [a * 100 for a in history["fa"]], label=f"FA ({classes[FORGET]} test)")
ax2.set(title="Retain vs forget accuracy", xlabel="epoch", ylabel="accuracy (%)", ylim=(-2, 100))
ax2.legend()
fig.tight_layout()
plt.savefig(RESULTS_DIR / "07_curves.png", dpi=120)
plt.show()

## 9. How expensive was this?

Retraining is the gold standard, but its cost is the whole point of the understanding note. Here we write it down so programs 08 and 09 can compare: they will take seconds to a minute, against the minutes of full training below. On a real model (ImageNet-scale, or a large language model) the same retraining would take days or weeks, for *every* deletion request.

In [ ]:
minutes = train_minutes
print(f"retraining took {minutes:.1f} minutes for {SETTINGS['epochs']} epochs on 45,000 images")

## 10. Log the run

In [ ]:
log_run("07_retrain_baseline", SETTINGS,
        {"ra": float(acc_retrain[retain_mask].mean()), "fa": float(acc_retrain[FORGET]),
         "per_class_acc": [round(float(a), 4) for a in acc_retrain],
         "forget_pred_share": [round(float(s), 4) for s in share],
         "conf_retain": float(conf_retain.mean()), "conf_forget": float(conf_forget.mean()),
         "minutes": round(minutes, 1)})
print("logged to", RESULTS_DIR / "runs.jsonl")

## Lab notebook

*Write one paragraph here: what you saw, and what surprised you.*

Questions to answer in it:
1. Is cat accuracy exactly 0%? Why is that guaranteed, and not just likely?
2. Did removing cats change the *other* classes? Look especially at dog: is it higher or lower than in the original model, and why? (Hint: in the original model, some dogs were mistaken for cats.)
3. What does the retrain model predict for cat images? Why is that pattern, and not "random class", the right target for an unlearning method?
4. Retrain confidence on cat images vs retain images: how big is the gap? How could an attacker use a gap like that to tell whether a model was trained on cats? (This is program 10.)
5. How long did retraining take? How often could you afford it if deletion requests arrived every hour?